# Python Agent Framework with ChromaDB Example

This document provides an overview and explanation of the code used to create a Python Agent Framework-based agent that integrates with ChromaDB for Retrieval-Augmented Generation (RAG). The example demonstrates how to build an AI agent that retrieves travel documents from a ChromaDB collection, augments user queries with semantic search results, and streams detailed travel recommendations.

## Initializing the Environment

SQLite Version Fix
If you encounter the error:
```
RuntimeError: Your system has an unsupported version of sqlite3. Chroma requires sqlite3 >= 3.35.0
```

Uncomment this code block at the start of your notebook:

In [ ]:
%pip install agent-framework-azure-ai chromadb -U

### Importing Packages
The following code imports the necessary packages:

In [ ]:
import json
import os
import chromadb

from typing import Annotated

from IPython.display import display, HTML

from dotenv import load_dotenv

from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

### Creating the Azure OpenAI Chat Client

An Azure OpenAI chat client is created and configured to connect to Azure AI Foundry. The client is used by the Python Agent Framework to generate responses.

In [5]:
load_dotenv()
# Load environment variables from .env file
print("Loading environment variables from .env file...")

# Print relevant environment variables (without exposing sensitive data)
azure_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
model_name = os.getenv("AZURE_AI_FOUNDRY_MODEL")
api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")

print(f"Azure Endpoint: {azure_endpoint}")
print(f"API Version: {api_version}")
print(f"Model Name: {model_name}")


Loading environment variables from .env file...
Azure Endpoint: https://ibeljan-foundry.openai.azure.com/
API Version: 2024-02-01
Model Name: gpt-4o


In [ ]:
# Configure the Azure AI Agent Client
if not api_key:
    raise ValueError(
        "AZURE_AI_FOUNDRY_API_KEY environment variable is required. "
        "Please set it in your .env file or environment."
    )

# Azure OpenAI Chat Client using Python Agent Framework
chat_client = AzureOpenAIChatClient(
    endpoint=azure_endpoint,
    api_key=api_key,
    deployment_name=model_name
)

### Defining the Tool Functions

Tool functions are defined to enable the agent to retrieve documents from ChromaDB and get weather information.

In [ ]:
# Dictionary of destinations and their average temperatures
destination_temperatures = {
    "maldives": "82°F (28°C)",
    "swiss alps": "45°F (7°C)",
    "african safaris": "75°F (24°C)"
}

def get_destination_temperature(destination: str) -> str:
    """Get the average temperature for a specific travel destination.
    
    Args:
        destination: The name of the travel destination
        
    Returns:
        The average temperature for the destination or an error message
    """
    # Normalize the input destination (lowercase)
    normalized_destination = destination.lower()
    
    # Look up the temperature for the destination
    if normalized_destination in destination_temperatures:
        return f"The average temperature in {destination} is {destination_temperatures[normalized_destination]}."
    else:
        return f"Sorry, I don't have temperature information for {destination}. Available destinations are: Maldives, Swiss Alps, and African safaris."

## ChromaDB Initialization

We initialize ChromaDB with persistent storage and add enhanced sample documents. ChromaDB will be used to store and retrieve documents that provide context for generating accurate responses.

In [ ]:
# Initialize ChromaDB with persistent storage
collection = chromadb.PersistentClient(path="./chroma_db").create_collection(
    name="travel_documents",
    metadata={"description": "travel_service"},
    get_or_create=True,
)

# Enhanced sample documents
documents = [
    "Contoso Travel offers luxury vacation packages to exotic destinations worldwide.",
    "Our premium travel services include personalized itinerary planning and 24/7 concierge support.",
    "Contoso's travel insurance covers medical emergencies, trip cancellations, and lost baggage.",
    "Popular destinations include the Maldives, Swiss Alps, and African safaris.",
    "Contoso Travel provides exclusive access to boutique hotels and private guided tours.",
]

# Add documents to the collection
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "training", "type": "explanation"} for _ in documents]
)

C:\Users\ivbeljan\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:05<00:00, 15.8MiB/s]


In [ ]:
# Define retrieval tool function
def retrieve_documents(query: str) -> str:
    """Retrieve documents from the ChromaDB collection based on the query.
    
    Args:
        query: The search query to find relevant documents
        
    Returns:
        A string containing retrieved document content or 'No results found'
    """
    results = collection.query(
        query_texts=[query],
        include=["documents", "metadatas"],
        n_results=2
    )
    context_strings = []
    if results and results.get("documents") and results["documents"][0]:
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            context_strings.append(f"Document: {doc}")
    return "\n\n".join(context_strings) if context_strings else "No results found"

In [ ]:
# Create the agent with tools
agent = chat_client.create_agent(
    name="TravelAgent",
    instructions="""
    You are a helpful travel assistant for Contoso Travel. 
    Answer travel queries using the provided tools and context.
    
    - Use the retrieve_documents tool to search for information about Contoso's services
    - Use the get_destination_temperature tool to find temperature information for destinations
    - Provide accurate, helpful responses based on the retrieved information
    - If you can't find relevant information, say so clearly
    """,
    tools=[retrieve_documents, get_destination_temperature],
    tool_choice="auto"
)

print("Agent created successfully!")

### Running the Agent with Streaming

The main asynchronous function runs the agent with multiple user queries. The agent uses streaming to provide real-time responses, and we capture both function calls and the final responses for display.

In [ ]:
async def main():
    user_inputs = [
        "Can you explain Contoso's travel insurance coverage?",
        "What is the average temperature of the Maldives?",
        "What is a good cold destination offered by Contoso and what is its average temperature?",
    ]

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response = []
        function_calls = []

        print(f"\n{'='*80}")
        print(f"User: {user_input}")
        print(f"{'='*80}\n")

        # Run the agent with streaming
        async for chunk in agent.run_stream(user_input):
            # Collect text responses
            if chunk.text:
                full_response.append(chunk.text)
                print(chunk.text, end="", flush=True)
            
            # Collect function call information if available
            if hasattr(chunk, 'tool_calls') and chunk.tool_calls:
                for tool_call in chunk.tool_calls:
                    function_calls.append(f"Calling function: {tool_call.function.name}({tool_call.function.arguments})")
            
            # Collect function results if available
            if hasattr(chunk, 'tool_outputs') and chunk.tool_outputs:
                for tool_output in chunk.tool_outputs:
                    function_calls.append(f"\nFunction Result:\n\n{tool_output}")

        print("\n")

        # Build HTML output for Jupyter display
        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()